In [ ]:
from dotenv import load_dotenv
from typing import TypedDict, List, Annotated
from langgraph.graph import StateGraph, START, END, add_messages
from langchain.chat_models import init_chat_model
from IPython.display import Image, display
from langgraph.checkpoint.memory import InMemorySaver  

load_dotenv()

In [ ]:
# initialize the LLM
llm = init_chat_model("llama-3.1-8b-instant", model_provider="groq")

In [ ]:
# graph state
class ChatState(TypedDict):
    messages: Annotated[list, add_messages]

# graph node function
def chatbot(state: ChatState)-> ChatState:
    return {"messages": [llm.invoke(state["messages"])]}

# build the graph by connecting nodes by edges
builder = StateGraph(ChatState)
builder.add_node("chatbot_node", chatbot)

builder.add_edge(START, "chatbot_node")
builder.add_edge("chatbot_node", END)

graph = builder.compile()
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
# calling LLM inline
msg = {"role":"user", "content": "how many states are there in India?"}
respState = graph.invoke({"messages": [msg]})
respState["messages"]

In [ ]:
# calling LLM inline
msg = {"role":"user", "content": "which is the largest state?"}
respState = graph.invoke({"messages": [msg]})
respState["messages"]

In [ ]:
# create graph with in-memory checkpointer
inMem = InMemorySaver()
graph_with_memory = builder.compile(checkpointer=inMem)

In [ ]:
# calling LLM inline with memory
chatMemIndia = {'configurable':{'thread_id': 'chat_thread_india'}}
msg = {"role":"user", "content": "how many states are there in India?"}
respState=graph_with_memory.invoke({"messages": [msg]},config=chatMemIndia)
respState["messages"]

In [ ]:
# calling LLM inline
msg = {"role":"user", "content": "which is the largest state?"}
respState = graph_with_memory.invoke({"messages": [msg]},config=chatMemIndia)
respState["messages"]

In [ ]:
# memory context chatbot listening to user input in a loop
chatState = None
while True:
    user_input = input("User: ")
    if user_input.lower() in ["exit", "quit"]:
        break
    chatState = graph_with_memory.invoke({"messages": [{"role": "user", "content": user_input}]},config=chatMemIndia)
    print("LLM:", chatState["messages"][-1].content)